# Week 2 — Market Environment Design & Custom Gym Environment
**SoC 2026 | Dynamic Pricing Using Reinforcement Learning**

**Topics Covered:**
- Reinforcement Learning framework (Agent-Environment loop)
- Linear demand function design
- State space, action space, reward function definition
- Building a custom OpenAI Gymnasium environment
- Unit testing the environment
- Analytical Nash price verification

**Resources studied (SOC Week 2 PDF):**
- Arxiv Insights: What is Reinforcement Learning?
- sentdex: OpenAI Gym Tutorial
- Nicholas Renotte: Building Custom Gym Environments
- EconplusDal: Linear Demand Functions
- Farama Foundation: Official Gymnasium Docs
- Sutton & Barto Ch. 1 & 2

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from src.environment.pricing_env import BertrandPricingEnv

## 1. Reinforcement Learning Framework

```
┌──────────────┐   action a_t    ┌─────────────────────┐
│              │ ──────────────► │                     │
│    AGENT     │                 │   ENVIRONMENT       │
│              │ ◄────────────── │  (Pricing Market)   │
└──────────────┘  obs s_{t+1}   └─────────────────────┘
                  reward r_t
```

**In our market:**
- **Agent** = pricing firm (sets price each period)
- **Environment** = Bertrand duopoly market
- **State** = previous prices of both firms (normalised)
- **Action** = discrete price choice from price grid
- **Reward** = normalised profit from current period

## 2. Demand Function Design

**Linear demand with cross-price effect:**
$$Q_i = a - b \cdot P_i + c \cdot P_j$$

- $a$ = demand intercept (market size)
- $b$ = own-price sensitivity (higher P_i → lower Q_i)
- $c$ = cross-price sensitivity (higher P_j → higher Q_i, substitute goods)
- Stability condition: $2b > c$

In [ ]:
# Visualise demand curves
a, b, c, mc = 10.0, 2.0, 0.5, 1.0

p_range = np.linspace(0, 8, 200)
p_opp_values = [2.0, 4.0, 6.0]  # different opponent prices

plt.figure(figsize=(8, 5))
for p_opp in p_opp_values:
    q = np.maximum(a - b * p_range + c * p_opp, 0)
    plt.plot(q, p_range, label=f"P_opponent = {p_opp}")

plt.axhline(mc, color='black', linestyle=':', label=f'Marginal Cost = {mc}')
plt.xlabel("Quantity Q_i")
plt.ylabel("Price P_i")
plt.title("Inverse Demand Curves — Bertrand Duopoly")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/figures/week2_demand_curves.png", dpi=150)
plt.show()

## 3. Environment Design Decisions

| Design Choice | Decision | Reason |
|---|---|---|
| Action space | Discrete, 50 price levels | Avoids Q-table explosion; ≥20 per SOC pitfall note |
| State space | (own_prev_price, opp_prev_price) normalised | Captures strategic interaction |
| Reward | Normalised profit ∈ [0,1] | Prevents Q-value explosion (SOC pitfall note) |
| Parameters | Configurable a,b,c,mc | Hardcoding is Pitfall #2 in Week 2 SOC notes |
| Episode length | 200 steps | Long enough for strategies to emerge |
| Price range | [mc, p_max] | mc = floor; prices below mc are irrational |

## 4. Building & Testing the Custom Environment

In [ ]:
# Instantiate environment
env = BertrandPricingEnv(a=10, b=2, c=0.5, mc=1.0, p_max=8.0, n_prices=50, seed=42)

print("=" * 50)
print("ENVIRONMENT SUMMARY")
print("=" * 50)
print(f"Action Space         : {env.action_space}")
print(f"Observation Space    : {env.observation_space}")
print(f"Price Grid (first 5) : {env.price_grid[:5].round(3)}")
print(f"Price Grid (last 5)  : {env.price_grid[-5:].round(3)}")
print(f"Nash Price  (P_nash) : {env.p_nash:.4f}")
print(f"Collude Price        : {env.p_collude:.4f}")
print(f"Nash Action Index    : {env.get_nash_action()}")
print(f"Collude Action Index : {env.get_collude_action()}")

In [ ]:
# Unit Test 1: reset() returns correct obs shape
obs, info = env.reset()
assert obs.shape == (2,), f"Expected shape (2,), got {obs.shape}"
assert 0.0 <= obs[0] <= 1.0, "obs[0] not normalised"
assert 0.0 <= obs[1] <= 1.0, "obs[1] not normalised"
print("✅ Test 1 PASS: reset() → obs shape and normalisation correct")

# Unit Test 2: step() returns correct structure
obs2, rewards, terminated, truncated, info = env.step([10, 20])
assert len(rewards) == 2, "Expected 2 rewards"
assert 0.0 <= rewards[0] <= 1.0, "Reward 0 not in [0,1]"
assert 0.0 <= rewards[1] <= 1.0, "Reward 1 not in [0,1]"
print("✅ Test 2 PASS: step() → rewards normalised, structure correct")

# Unit Test 3: Nash action gives Nash price
nash_action = env.get_nash_action()
nash_price_from_action = env.price_grid[nash_action]
assert abs(nash_price_from_action - env.p_nash) < 0.5, "Nash action mapping error"
print(f"✅ Test 3 PASS: Nash action {nash_action} → price {nash_price_from_action:.3f} ≈ P_nash {env.p_nash:.3f}")

# Unit Test 4: episode terminates at max_steps
env2 = BertrandPricingEnv(max_steps=5)
obs, _ = env2.reset()
for _ in range(5):
    obs, rew, term, trunc, info = env2.step([0, 0])
assert trunc == True, "Episode should have truncated at step 5"
print("✅ Test 4 PASS: Episode truncation at max_steps correct")

print("\n🎉 All unit tests passed!")

## 5. Profit Function Visualisation

In [ ]:
# Show profit of Firm 1 vs its own price, holding Firm 2 at different fixed prices
env = BertrandPricingEnv()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Firm 1 profit vs P1 for various P2
p1_range = np.linspace(mc, 8, 100)
for p2_fixed in [env.p_nash, env.p_collude, 3.0]:
    profits = [env._profit(p1, p2_fixed) for p1 in p1_range]
    axes[0].plot(p1_range, profits, label=f"P2={p2_fixed:.2f}")
axes[0].axvline(env.p_nash,    color='red',   linestyle='--', label=f"P_Nash={env.p_nash:.2f}")
axes[0].axvline(env.p_collude, color='green', linestyle='--', label=f"P_Collude={env.p_collude:.2f}")
axes[0].set_xlabel("Firm 1 Price")
axes[0].set_ylabel("Firm 1 Profit")
axes[0].set_title("Firm 1 Profit vs Own Price")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Right: Normalised reward signal across price grid
p2_fixed = env.p_collude
raw_profits    = [env._profit(p, p2_fixed) for p in env.price_grid]
norm_rewards   = [min(max(pr / (env._max_profit + 1e-8), 0), 1) for pr in raw_profits]
axes[1].plot(env.price_grid, norm_rewards, color='purple')
axes[1].axvline(env.p_nash,    color='red',   linestyle='--', label=f"P_Nash")
axes[1].axvline(env.p_collude, color='green', linestyle='--', label=f"P_Collude")
axes[1].set_xlabel("Firm 1 Price (action)")
axes[1].set_ylabel("Normalised Reward")
axes[1].set_title(f"Normalised Reward Signal (P2={p2_fixed:.2f})")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/figures/week2_profit_reward.png", dpi=150)
plt.show()

## 6. Random Episode Walkthrough

In [ ]:
# Run one random episode and visualise prices and rewards
env = BertrandPricingEnv(max_steps=100, seed=42)
obs, _ = env.reset()
rng = np.random.default_rng(0)

step_prices_0, step_prices_1, step_rewards_0 = [], [], []
done = False

while not done:
    a0 = int(rng.integers(0, env.n_prices))
    a1 = int(rng.integers(0, env.n_prices))
    obs, rewards, term, trunc, info = env.step([a0, a1])
    step_prices_0.append(info['price_0'])
    step_prices_1.append(info['price_1'])
    step_rewards_0.append(rewards[0])
    done = term or trunc

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(step_prices_0, label='Firm 0', alpha=0.8)
axes[0].plot(step_prices_1, label='Firm 1', alpha=0.8)
axes[0].axhline(env.p_nash,    color='red',   linestyle='--', label=f'Nash={env.p_nash:.2f}')
axes[0].axhline(env.p_collude, color='green', linestyle='--', label=f'Collude={env.p_collude:.2f}')
axes[0].set_ylabel('Price'); axes[0].set_title('Random Episode — Price Trajectory'); axes[0].legend()

axes[1].plot(step_rewards_0, color='purple', alpha=0.8)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Normalised Reward (Firm 0)')
axes[1].set_title('Normalised Reward per Step')

plt.tight_layout()
plt.savefig("../results/figures/week2_random_episode.png", dpi=150)
plt.show()

## 7. Week 2 Summary

| Component | Implementation |
|---|---|
| Demand | Q_i = 10 - 2*P_i + 0.5*P_j |
| Action Space | Discrete(50) — prices in [1.0, 8.0] |
| State Space | Box(2,) — normalised prev prices ∈ [0,1]² |
| Reward | Normalised profit ∈ [0,1] |
| Nash Price | 3.67 |
| Collude Price | 4.00 |
| All unit tests | ✅ Passed |

**Next Week:** Implement rule-based baseline agents and run tournament.